In [1]:
from cropspectai import ml_config
from cropspectai.data_analysis import preprocessing

import pandas as pd

In [2]:
def clip_outliers_drop_duplicates_and_save(df, target, threshold, datasplit):
    # clip values above threshold
    df[target] = preprocessing.clip_values_above_threshold(df, target, threshold).copy() 
    print("Shape after clipping: ", df.shape)

    # DROP duplicates based on Nan in target
    df = preprocessing.drop_duplicates(df, target).copy()
    print("Shape after dropping duplicates: ", df.shape)

    output_path = ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH / target / f"{datasplit}.parquet"

    df.to_parquet(
        output_path,
        engine="pyarrow",
        index=False,
    )

# Train data

In [3]:
train_path = ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH / "all" / "train.parquet"
df_train = pd.read_parquet(train_path)

print(df_train.columns)
print(df_train.shape)

target = "cab"

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'cab', 'cw', 'cm', 'LAI', 'cp', 'cbc', 'car', 'anth'],
      dtype='str')
(3550, 20)


## check for outlier in target (cab)

In [4]:
df_train[target].describe(percentiles=[0.95, 0.99, 0.995])

count    1607.000000
mean       37.887584
std        15.145647
min         5.164474
95%        63.383701
99%        71.277162
99.5%      73.730804
max       229.497477
Name: cab, dtype: float64

### find sample count above threshold

In [5]:
preprocessing.find_sample_count_above_threshold(df_train, target, thresholds=[40, 60, 80, 100, 150, 200]) 

cab > 40: 648 samples
cab > 60: 131 samples
cab > 80: 3 samples
cab > 100: 2 samples
cab > 150: 1 samples
cab > 200: 1 samples


## clean and save

In [6]:
clip_outliers_drop_duplicates_and_save(df_train, target, threshold=100, datasplit = "train")

Shape after clipping:  (3550, 20)
(Before) Shape of dataframe: (3550, 20)
Dropping duplicate rows
Duplicate rows: 0
(After) Shape of dataframe: (3550, 20)
Checking for rows without labels
Rows with no labels: 1943 with target: cab
(After) Shape of dataframe: (1607, 13)
Shape after dropping duplicates:  (1607, 13)


# Test data

In [7]:
test_path = ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH / "all" / "test.parquet"
df_test = pd.read_parquet(test_path)

print(df_test.columns)
print(df_test.shape)

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'cab', 'cw', 'cm', 'LAI', 'cp', 'cbc', 'car', 'anth'],
      dtype='str')
(883, 20)


## check for outlier in target (cab)

In [8]:
df_test[target].describe(percentiles=[0.95, 0.99, 0.995])

count    396.000000
mean      37.838184
std       13.576089
min        4.448305
95%       60.693731
99%       67.004826
99.5%     69.606474
max       78.123940
Name: cab, dtype: float64

In [9]:
preprocessing.find_sample_count_above_threshold(df_test, target, thresholds=[40, 60, 80, 100, 150, 200]) 

cab > 40: 164 samples
cab > 60: 22 samples
cab > 80: 0 samples
cab > 100: 0 samples
cab > 150: 0 samples
cab > 200: 0 samples


## clean and save

In [10]:
clip_outliers_drop_duplicates_and_save(df_test, target, threshold=100, datasplit = "test")

Shape after clipping:  (883, 20)
(Before) Shape of dataframe: (883, 20)
Dropping duplicate rows
Duplicate rows: 0
(After) Shape of dataframe: (883, 20)
Checking for rows without labels
Rows with no labels: 487 with target: cab
(After) Shape of dataframe: (396, 13)
Shape after dropping duplicates:  (396, 13)
